# Data Ingestion

## Use JDBC
NOTE: Add the Databricks Workspace IP to the Master Database firewall.

Connect SQL Server Database to the notebook to get the **transactions_data** and **cars_data** table.

In [0]:
user = "bellemin_alex"
password = "rocky1234!"
database = "tic"
url = f"jdbc:sqlserver://jrvs-etl-databricks-alexandre.database.windows.net:1433;encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
driver = "com.microsoft.sqlserver.jdbc.SQLServerDriver"

transactions_df = (spark.read
    .format("jdbc")
    .option("url", url)
    .option("database", database)
    .option("dbtable", "transactions_data")
    .option("user", user)
    .option("password", password)
    .option("driver", driver)
    .load()
)
display(transactions_df)

In [0]:
cards_df = (spark.read
    .format("jdbc")
    .option("url", url)
    .option("database", database)
    .option("dbtable", "cards_data")
    .option("user", user)
    .option("password", password)
    .option("driver", driver)
    .load()
)
display(cards_df)

## Use Unity Catalog
We can read the CSV in the Volumes for **users_data**.

In [0]:
users_df = (
    spark.read
    .csv("/Volumes/jarvis_tic/external_tables/external_json/users_data.csv", header=True, inferSchema=True)
)
display(users_df)

## Get the JSON files
They enhance the silver transaction table 

In [0]:
from pyspark.sql import functions as F
# Import JSON to dataframe
mcc_raw = (
    spark.read
    .option("multiline", "true")
    .option("mergeSchema", "true")
    .json("/Volumes/jarvis_tic/external_tables/external_json/mcc_codes.json")
)

# Transform the struct in map<string,string>
mcc_map = mcc_raw.select(
    F.create_map(
        *[
            x
            for field in mcc_raw.columns
            for x in (F.lit(field), F.col(f"`{field}`"))
        ]
    ).alias("mcc_map")
)

# Explode the map
mcc_codes_df = mcc_map.select(
    F.explode("mcc_map").alias("code", "value")
)

display(mcc_codes_df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, MapType, StringType

# Define schema to force target as MapType
schema = StructType([
    StructField(
        "target",
        MapType(StringType(), StringType()),
        True
    )
])

# Read JSON using explicit schema
train_raw = (
    spark.read
    .option("multiline", "true")
    .option("mergeSchema", "true")
    .schema(schema)
    .json("/Volumes/jarvis_tic/external_tables/external_json/train_fraud_labels.json")
)

# Explode directly (no struct to map conversion needed anymore like mcc codes)
train_df = train_raw.select(
    F.explode("target").alias("id", "value")
)

display(train_df)

## Save to Bronze Table
Now that the data has been ingested, we need to save them in our Bronze schema in order to perform transformations on the data.

In [0]:
# Write your raw data to a table
transactions_df.write.mode("overwrite").saveAsTable("jarvis_tic.01_bronze.transactions_data")
cards_df.write.mode("overwrite").saveAsTable("jarvis_tic.01_bronze.cards_data")
users_df.write.mode("overwrite").saveAsTable("jarvis_tic.01_bronze.users_data")
mcc_codes_df.write.mode("overwrite").saveAsTable("jarvis_tic.01_bronze.mcc_codes_data")
train_df.write.mode("overwrite").saveAsTable("jarvis_tic.01_bronze.train_fraud_labels_data")